In [2]:
import pandas as pd

# path to one raw file — adjust if your data lives elsewhere
RAW = "data/record_2019-01-01.csv"

# read only the columns we need
df = pd.read_csv(RAW, usecols=["time", "stationID", "status"], parse_dates=["time"])

# floor each timestamp down to its 15-minute bucket
df["interval"] = df["time"].dt.floor("15min")

print(df.shape)
print(df.head())
print(df["interval"].nunique(), "distinct 15-min buckets")

(2539592, 4)
                 time  stationID  status            interval
0 2019-01-01 02:00:05         27       0 2019-01-01 02:00:00
1 2019-01-01 02:01:40          5       1 2019-01-01 02:00:00
2 2019-01-01 02:01:53          5       0 2019-01-01 02:00:00
3 2019-01-01 02:02:38          5       0 2019-01-01 02:00:00
4 2019-01-01 02:03:42         23       0 2019-01-01 02:00:00
88 distinct 15-min buckets


In [3]:
# count taps per (station, interval, status)
counts = (
    df.groupby(["stationID", "interval", "status"])
      .size()
      .reset_index(name="count")
)

print(counts.head(10))
print(counts.shape)

   stationID            interval  status  count
0          0 2019-01-01 05:30:00       1      3
1          0 2019-01-01 05:45:00       0      1
2          0 2019-01-01 05:45:00       1      5
3          0 2019-01-01 06:00:00       1     28
4          0 2019-01-01 06:15:00       0      1
5          0 2019-01-01 06:15:00       1     40
6          0 2019-01-01 06:30:00       0     89
7          0 2019-01-01 06:30:00       1     62
8          0 2019-01-01 06:45:00       0     33
9          0 2019-01-01 06:45:00       1     79
(11739, 4)


In [4]:
# pivot status (0/1) from rows into columns
flow = counts.pivot_table(
    index=["stationID", "interval"],
    columns="status",
    values="count",
    fill_value=0
).reset_index()

# rename the status columns to meaningful names
flow = flow.rename(columns={0: "inflow", 1: "outflow"})
flow.columns.name = None  # tidy up the leftover 'status' label

print(flow.head(10))
print(flow.shape)

   stationID            interval  inflow  outflow
0          0 2019-01-01 05:30:00     0.0      3.0
1          0 2019-01-01 05:45:00     1.0      5.0
2          0 2019-01-01 06:00:00     0.0     28.0
3          0 2019-01-01 06:15:00     1.0     40.0
4          0 2019-01-01 06:30:00    89.0     62.0
5          0 2019-01-01 06:45:00    33.0     79.0
6          0 2019-01-01 07:00:00    35.0    150.0
7          0 2019-01-01 07:15:00    39.0    145.0
8          0 2019-01-01 07:30:00    39.0    145.0
9          0 2019-01-01 07:45:00    46.0    183.0
(6015, 4)


In [5]:
flow[["inflow", "outflow"]] = flow[["inflow", "outflow"]].astype(int)

In [6]:
import glob
import os

def aggregate_one_day(path):
    df = pd.read_csv(path, usecols=["time", "stationID", "status"], parse_dates=["time"])
    df["interval"] = df["time"].dt.floor("15min")
    counts = (
        df.groupby(["stationID", "interval", "status"])
          .size()
          .reset_index(name="count")
    )
    flow = counts.pivot_table(
        index=["stationID", "interval"],
        columns="status", values="count", fill_value=0
    ).reset_index()
    flow = flow.rename(columns={0: "inflow", 1: "outflow"})
    flow.columns.name = None
    flow[["inflow", "outflow"]] = flow[["inflow", "outflow"]].astype(int)
    return flow

# find all daily record files, sorted so they're in date order
files = sorted(glob.glob("data/record_2019-01-*.csv"))
print(f"Found {len(files)} files")

# process each, collect results
daily_tables = []
for path in files:
    day = aggregate_one_day(path)
    daily_tables.append(day)
    print(f"  {os.path.basename(path)} -> {day.shape}")

# stack all days into one table
flow_all = pd.concat(daily_tables, ignore_index=True)
print("Combined:", flow_all.shape)

Found 25 files
  record_2019-01-01.csv -> (6015, 4)
  record_2019-01-02.csv -> (6124, 4)
  record_2019-01-03.csv -> (6111, 4)
  record_2019-01-04.csv -> (6072, 4)
  record_2019-01-05.csv -> (6130, 4)
  record_2019-01-06.csv -> (6017, 4)
  record_2019-01-07.csv -> (6059, 4)
  record_2019-01-08.csv -> (6077, 4)
  record_2019-01-09.csv -> (6157, 4)
  record_2019-01-10.csv -> (6122, 4)
  record_2019-01-11.csv -> (6100, 4)
  record_2019-01-12.csv -> (6070, 4)
  record_2019-01-13.csv -> (6069, 4)
  record_2019-01-14.csv -> (6047, 4)
  record_2019-01-15.csv -> (6113, 4)
  record_2019-01-16.csv -> (6054, 4)
  record_2019-01-17.csv -> (6162, 4)
  record_2019-01-18.csv -> (6091, 4)
  record_2019-01-19.csv -> (6102, 4)
  record_2019-01-20.csv -> (6068, 4)
  record_2019-01-21.csv -> (6138, 4)
  record_2019-01-22.csv -> (6187, 4)
  record_2019-01-23.csv -> (6233, 4)
  record_2019-01-24.csv -> (6182, 4)
  record_2019-01-25.csv -> (6222, 4)
Combined: (152722, 4)


In [7]:
os.makedirs("data/processed", exist_ok=True)
flow_all.to_csv("data/processed/flow_15min.csv", index=False)
print("Saved:", flow_all.shape)
print(flow_all.head())

Saved: (152722, 4)
   stationID            interval  inflow  outflow
0          0 2019-01-01 05:30:00       0        3
1          0 2019-01-01 05:45:00       1        5
2          0 2019-01-01 06:00:00       0       28
3          0 2019-01-01 06:15:00       1       40
4          0 2019-01-01 06:30:00      89       62
